# 22. Ensemble Learning: Gradient Boosting

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: Medium-High  
**Use Case**: Sequential boosting using gradient descent to minimize loss

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Gradient Boosting algorithm and how it differs from AdaBoost
- Implement Gradient Boosting for classification and regression
- Understand how gradient descent is used in boosting
- Visualize how the model improves over iterations
- Tune hyperparameters (n_estimators, learning_rate, max_depth)
- Apply Gradient Boosting to real-world problems

## Historical Context

Gradient Boosting was developed by Jerome Friedman in 2001:
- Friedman, J.H. (2001): "Greedy Function Approximation: A Gradient Boosting Machine"
- Generalization of boosting using gradient descent
- Foundation for XGBoost, LightGBM, and CatBoost

**Key Papers/References:**
- Friedman, J.H. (2001). "Greedy Function Approximation: A Gradient Boosting Machine"
- Friedman, J.H. (2002). "Stochastic Gradient Boosting"

## When to Use Gradient Boosting

Gradient Boosting is appropriate when:
- You need high accuracy
- Working with structured/tabular data
- Non-linear relationships in data
- You have time for hyperparameter tuning
- Moderate to large datasets
- Both classification and regression tasks

## Theory & Mechanics

### Mathematical Foundation

Gradient Boosting uses gradient descent to sequentially add weak learners that minimize the loss function.

**Initialization:**
$$F_0(x) = \arg\min_{\gamma} \sum_{i=1}^{N} L(y_i, \gamma)$$

**For each iteration t = 1, 2, ..., T:**

1. **Calculate residuals (negative gradients):**
   $$r_{it} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x)=F_{t-1}(x)}$$

2. **Fit weak learner to residuals:**
   $$h_t(x) = \arg\min_{h} \sum_{i=1}^{N} (r_{it} - h(x_i))^2$$

3. **Find optimal step size:**
   $$\gamma_t = \arg\min_{\gamma} \sum_{i=1}^{N} L(y_i, F_{t-1}(x_i) + \gamma h_t(x_i))$$

4. **Update model:**
   $$F_t(x) = F_{t-1}(x) + \gamma_t h_t(x)$$

**Final Prediction:**
$$\hat{y} = F_T(x) = F_0(x) + \sum_{t=1}^{T} \gamma_t h_t(x)$$

### How It Works

1. **Start**: Initialize with constant prediction (mean for regression, log-odds for classification)
2. **Calculate gradients**: Compute negative gradients (residuals) of loss function
3. **Fit weak learner**: Train decision tree to predict residuals
4. **Update model**: Add weak learner to ensemble with optimal step size
5. **Repeat**: Continue until convergence or max iterations

### Key Hyperparameters

- **n_estimators**: Number of boosting iterations
- **learning_rate**: Shrinks contribution of each tree (default: 0.1)
- **max_depth**: Maximum depth of weak learners (default: 3)
- **min_samples_split**: Minimum samples to split a node
- **subsample**: Fraction of samples to use for each tree (stochastic gradient boosting)
- **loss**: Loss function ('deviance' for classification, 'ls' for regression)

### Advantages

- High predictive accuracy
- Handles non-linear relationships well
- Flexible (can use different loss functions)
- Provides feature importance
- Works well with default parameters

### Limitations

- Sequential training (cannot parallelize easily)
- Sensitive to overfitting
- Requires careful hyperparameter tuning
- Can be slow for large datasets
- Memory intensive


## Implementation

Let's implement Gradient Boosting for both classification and regression.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.models.ensemble import extract_feature_importance, plot_feature_importance
from src.utils.benchmarking import benchmark_model_training
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Classification Example: Breast Cancer
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='Target')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {cancer.target_names.tolist()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting Classifier
model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model.fit(X_train, y_train)

print("\nGradient Boosting Classifier:")
print(f"Number of estimators: {model.n_estimators}")
print(f"Learning rate: {model.learning_rate}")
print(f"Max depth: {model.max_depth}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Learning Curve

Let's visualize how the model improves over iterations.


In [ ]:
# Use staged_predict to get predictions at each stage
train_scores = []
test_scores = []

for train_pred, test_pred in zip(
    model.staged_predict(X_train),
    model.staged_predict(X_test)
):
    train_scores.append(accuracy_score(y_train, train_pred))
    test_scores.append(accuracy_score(y_test, test_pred))

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_scores) + 1), train_scores, 'o-', label='Training Accuracy', markersize=3)
plt.plot(range(1, len(test_scores) + 1), test_scores, 's-', label='Test Accuracy', markersize=3)
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting: Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check for overfitting
overfitting_gap = [t - s for t, s in zip(train_scores, test_scores)]
if max(overfitting_gap) > 0.1:
    print(f"Warning: Potential overfitting detected (max gap: {max(overfitting_gap):.3f})")
else:
    print(f"Model shows good generalization (max gap: {max(overfitting_gap):.3f})")


## Validation & Testing

Let's validate the model and check for overfitting.


In [ ]:
# Validation 1: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 2: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > 0.5, "Accuracy should be better than random!"
print("\n✓ Validation checks passed")


## Feature Importance

Let's extract and visualize feature importance.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize
plot_feature_importance(feature_importance, top_n=15, title="Gradient Boosting Feature Importance")


## Regression Example

Let's apply Gradient Boosting to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Gradient Boosting Regressor
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = gb_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("Gradient Boosting Regression:")
print(f"  Test RMSE: {rmse:.3f}")

# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.6)
plt.plot([y_reg_test.min(), y_reg_test.max()], 
         [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Gradient Boosting Regression: Predicted vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Real-World Application

Let's tune hyperparameters and compare learning rates.


In [ ]:
# Compare different learning rates
learning_rates = [0.01, 0.1, 0.5, 1.0]
lr_results = {}

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    acc = accuracy_score(y_test, pred)
    lr_results[lr] = acc
    print(f"Learning rate {lr}: Test Accuracy = {acc:.3f}")

best_lr = max(lr_results, key=lr_results.get)
print(f"\nBest learning rate: {best_lr} with accuracy: {lr_results[best_lr]:.3f}")

# Hyperparameter tuning (reduced grid for speed)
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Gradient Boosting Basics**
   - Sequential ensemble using gradient descent
   - Fits weak learners to residuals (negative gradients)
   - Minimizes loss function iteratively
   - More flexible than AdaBoost (works with any differentiable loss)

2. **Gradient Descent in Boosting**
   - Calculate gradients of loss function
   - Fit weak learner to negative gradients
   - Update model by adding weak learner
   - Repeat until convergence

3. **Key Hyperparameters**
   - **n_estimators**: Number of boosting iterations
   - **learning_rate**: Shrinks contribution (lower = more iterations needed)
   - **max_depth**: Controls complexity of weak learners
   - **subsample**: Fraction of samples per tree (stochastic boosting)

4. **Best Practices**
   - Use small learning rate (0.1) with more estimators
   - Monitor learning curve to detect overfitting
   - Use early stopping if available
   - Tune max_depth to control complexity

### When to Use Gradient Boosting

✅ **Good for:**
- High accuracy requirements
- Structured/tabular data
- Non-linear relationships
- Both classification and regression
- When you can tune hyperparameters

❌ **Not ideal for:**
- Very large datasets (use XGBoost or LightGBM)
- Real-time predictions (can be slow)
- When interpretability is crucial
- High-dimensional sparse data
- When parallelization is critical

### Next Steps

- Try **XGBoost** for optimized gradient boosting
- Explore **LightGBM** for faster training
- Consider **CatBoost** for categorical features
- Use **Early Stopping** to prevent overfitting
- Compare with **Random Forest** (bagging vs boosting)
